# StreetLeague — Système de Recommandation de Terrains

## Objectif
Recommander des terrains (endroits) personnalisés à chaque client en se basant sur :
- **Historique des réservations** : quels terrains le client a déjà réservés
- **Disponibilité** : on ne recommande que les terrains disponibles

## Approche
On utilise le **Collaborative Filtering** avec l'algorithme **SVD** (Singular Value Decomposition).
L'idée : si deux clients ont des historiques similaires, on recommande à l'un ce que l'autre a aimé.

## Pipeline
1. Construction du dataset
2. Entraînement du modèle SVD
3. Évaluation (RMSE, MAE)
4. Exposition en web service Flask

## Étape 1 — Installation des dépendances

In [ ]:
# On installe scikit-surprise pour le collaborative filtering
# et les autres libs nécessaires
!pip install scikit-surprise pandas numpy matplotlib seaborn joblib

## Étape 2 — Construction du Dataset

Le dataset représente les interactions **client × terrain**.
Chaque ligne = un client a réservé un terrain.

On calcule un **score implicite** basé sur :
- Nombre de fois que le client a réservé ce terrain
- Statut de la réservation (CONFIRMEE = 5, EN_ATTENTE = 3, ANNULEE = 1)

Ici on génère un dataset synthétique réaliste. En production, on le remplace par les vraies données MySQL.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

np.random.seed(42)

# --- Terrains disponibles (sous-espaces) ---
terrains = [
    {"id": 1, "nom": "Terrain Football A",  "type": "FOOTBALL",   "disponible": True,  "prix_base": 20},
    {"id": 2, "nom": "Terrain Tennis 1",    "type": "TENNIS",     "disponible": True,  "prix_base": 15},
    {"id": 3, "nom": "Terrain Basketball",  "type": "BASKETBALL", "disponible": True,  "prix_base": 18},
    {"id": 4, "nom": "Terrain Football B",  "type": "FOOTBALL",   "disponible": False, "prix_base": 20},
    {"id": 5, "nom": "Terrain Padel",       "type": "PADEL",      "disponible": True,  "prix_base": 25},
    {"id": 6, "nom": "Terrain Volleyball",  "type": "VOLLEYBALL", "disponible": True,  "prix_base": 12},
    {"id": 7, "nom": "Terrain Tennis 2",    "type": "TENNIS",     "disponible": False, "prix_base": 15},
    {"id": 8, "nom": "Terrain Futsal",      "type": "FUTSAL",     "disponible": True,  "prix_base": 22},
]

df_terrains = pd.DataFrame(terrains)

# --- Génération des réservations (historique) ---
# 50 clients, chacun a réservé entre 2 et 8 terrains
n_clients = 50
statut_scores = {"CONFIRMEE": 5, "EN_ATTENTE": 3, "ANNULEE": 1}

rows = []
for client_id in range(1, n_clients + 1):
    # Chaque client a des préférences de type
    preferred_types = np.random.choice(["FOOTBALL", "TENNIS", "BASKETBALL", "PADEL", "FUTSAL"], 
                                        size=2, replace=False)
    n_reservations = np.random.randint(2, 9)
    
    for _ in range(n_reservations):
        # 70% chance de choisir un terrain du type préféré
        if np.random.random() < 0.7:
            terrain_pool = df_terrains[df_terrains["type"].isin(preferred_types)]
        else:
            terrain_pool = df_terrains
        
        terrain = terrain_pool.sample(1).iloc[0]
        statut = np.random.choice(["CONFIRMEE", "EN_ATTENTE", "ANNULEE"], p=[0.6, 0.3, 0.1])
        score = statut_scores[statut]
        
        rows.append({
            "client_id": client_id,
            "terrain_id": terrain["id"],
            "terrain_nom": terrain["nom"],
            "statut": statut,
            "score": score
        })

df_reservations = pd.DataFrame(rows)

# Agréger : si un client a réservé plusieurs fois le même terrain, on prend le score max
df_interactions = df_reservations.groupby(["client_id", "terrain_id"])["score"].max().reset_index()

print(f"Dataset: {len(df_interactions)} interactions, {df_interactions['client_id'].nunique()} clients, {df_interactions['terrain_id'].nunique()} terrains")
df_interactions.head(10)

## Étape 3 — Exploration du Dataset

Avant d'entraîner, on visualise les données pour comprendre leur distribution.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Distribution des scores
df_interactions["score"].value_counts().sort_index().plot(kind="bar", ax=axes[0], color="#22c55e")
axes[0].set_title("Distribution des scores")
axes[0].set_xlabel("Score (1=Annulée, 3=En attente, 5=Confirmée)")
axes[0].set_ylabel("Nombre d'interactions")

# Popularité des terrains
terrain_pop = df_interactions.groupby("terrain_id").size().reset_index(name="nb_reservations")
terrain_pop = terrain_pop.merge(df_terrains[["id", "nom"]], left_on="terrain_id", right_on="id")
terrain_pop.plot(kind="bar", x="nom", y="nb_reservations", ax=axes[1], color="#3b82f6", legend=False)
axes[1].set_title("Popularité des terrains")
axes[1].set_xlabel("")
axes[1].tick_params(axis='x', rotation=45)

# Activité des clients
client_activity = df_interactions.groupby("client_id").size()
axes[2].hist(client_activity, bins=10, color="#f59e0b", edgecolor="white")
axes[2].set_title("Activité des clients")
axes[2].set_xlabel("Nombre de terrains réservés")
axes[2].set_ylabel("Nombre de clients")

plt.tight_layout()
plt.savefig("exploration.png", dpi=100, bbox_inches="tight")
plt.show()
print("Graphiques sauvegardés dans exploration.png")

## Étape 4 — Entraînement du Modèle SVD

**SVD (Singular Value Decomposition)** décompose la matrice client×terrain en facteurs latents.
Ces facteurs capturent les préférences cachées des clients (ex: préfère les terrains couverts, préfère le football...).

On utilise la librairie `surprise` qui est spécialisée pour les systèmes de recommandation.

In [ ]:
from surprise import SVD, Dataset, Reader
from surprise.model_selection import cross_validate, train_test_split
from surprise import accuracy

# Préparer les données pour surprise
# Reader définit l'échelle des scores (1 à 5)
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df_interactions[["client_id", "terrain_id", "score"]], reader)

# Split train/test (80/20)
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

# Modèle SVD avec hyperparamètres optimisés
# n_factors: nombre de facteurs latents (dimensions cachées)
# n_epochs: nombre d'itérations d'entraînement
# lr_all: learning rate
# reg_all: régularisation pour éviter l'overfitting
model = SVD(n_factors=50, n_epochs=30, lr_all=0.005, reg_all=0.02, random_state=42)
model.fit(trainset)

print("Modèle SVD entraîné avec succès !")
print(f"Facteurs latents: 50 | Epochs: 30 | LR: 0.005 | Reg: 0.02")

## Étape 5 — Évaluation du Modèle

On évalue avec deux métriques :
- **RMSE** (Root Mean Square Error) : erreur quadratique moyenne — plus c'est bas, mieux c'est
- **MAE** (Mean Absolute Error) : erreur absolue moyenne

On fait aussi une **cross-validation 5-fold** pour avoir une évaluation robuste.

In [ ]:
# Évaluation sur le test set
predictions = model.test(testset)
rmse = accuracy.rmse(predictions)
mae  = accuracy.mae(predictions)

print(f"\n--- Résultats sur le test set ---")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")

# Cross-validation 5-fold
print("\n--- Cross-validation 5-fold ---")
cv_results = cross_validate(model, data, measures=["RMSE", "MAE"], cv=5, verbose=True)

print(f"\nRMSE moyen : {cv_results['test_rmse'].mean():.4f} ± {cv_results['test_rmse'].std():.4f}")
print(f"MAE moyen  : {cv_results['test_mae'].mean():.4f} ± {cv_results['test_mae'].std():.4f}")

In [ ]:
# Visualisation des erreurs de prédiction
pred_df = pd.DataFrame([(p.uid, p.iid, p.r_ui, p.est) for p in predictions],
                        columns=["client_id", "terrain_id", "score_reel", "score_predit"])
pred_df["erreur"] = abs(pred_df["score_reel"] - pred_df["score_predit"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scatter: réel vs prédit
axes[0].scatter(pred_df["score_reel"], pred_df["score_predit"], alpha=0.5, color="#22c55e")
axes[0].plot([1, 5], [1, 5], "r--", label="Prédiction parfaite")
axes[0].set_xlabel("Score réel")
axes[0].set_ylabel("Score prédit")
axes[0].set_title("Score réel vs prédit")
axes[0].legend()

# Distribution des erreurs
axes[1].hist(pred_df["erreur"], bins=20, color="#3b82f6", edgecolor="white")
axes[1].set_xlabel("Erreur absolue")
axes[1].set_ylabel("Fréquence")
axes[1].set_title(f"Distribution des erreurs (MAE={mae:.3f})")

plt.tight_layout()
plt.savefig("evaluation.png", dpi=100, bbox_inches="tight")
plt.show()

## Étape 6 — Fonction de Recommandation avec Filtre Disponibilité

On prédit le score pour tous les terrains que le client n'a **pas encore réservés**,
puis on **filtre les terrains non disponibles** avant de retourner le top-N.

In [ ]:
def recommander_terrains(client_id, model, df_terrains, df_interactions, top_n=5):
    """
    Recommande les top_n terrains pour un client donné.
    
    Étapes :
    1. Identifier les terrains déjà réservés par le client
    2. Prédire le score pour les terrains non réservés ET disponibles
    3. Trier par score décroissant et retourner le top-N
    """
    # Terrains déjà réservés par ce client
    deja_reserves = set(df_interactions[df_interactions["client_id"] == client_id]["terrain_id"])
    
    # Terrains disponibles non encore réservés
    terrains_candidats = df_terrains[
        (df_terrains["disponible"] == True) & 
        (~df_terrains["id"].isin(deja_reserves))
    ]
    
    if terrains_candidats.empty:
        return []
    
    # Prédire le score pour chaque terrain candidat
    recommendations = []
    for _, terrain in terrains_candidats.iterrows():
        pred = model.predict(client_id, terrain["id"])
        recommendations.append({
            "terrain_id": terrain["id"],
            "nom": terrain["nom"],
            "type": terrain["type"],
            "prix_base": terrain["prix_base"],
            "disponible": terrain["disponible"],
            "score_predit": round(pred.est, 3)
        })
    
    # Trier par score décroissant
    recommendations.sort(key=lambda x: x["score_predit"], reverse=True)
    return recommendations[:top_n]


# Test pour le client 1
client_test = 1
recs = recommander_terrains(client_test, model, df_terrains, df_interactions, top_n=5)

print(f"Recommandations pour le client {client_test}:")
print(f"Déjà réservés: {set(df_interactions[df_interactions['client_id']==client_test]['terrain_id'].tolist())}")
print()
for i, r in enumerate(recs, 1):
    print(f"{i}. {r['nom']} ({r['type']}) — Score prédit: {r['score_predit']} — {r['prix_base']} DT/h")

## Étape 7 — Sauvegarde du Modèle

On sauvegarde le modèle entraîné avec `joblib` pour le charger dans le web service Flask.

In [ ]:
import joblib
import json

# Sauvegarder le modèle SVD
joblib.dump(model, "svd_model.pkl")
print("Modèle sauvegardé : svd_model.pkl")

# Sauvegarder les interactions pour le web service
df_interactions.to_csv("interactions.csv", index=False)
print("Interactions sauvegardées : interactions.csv")

# Sauvegarder les terrains
df_terrains.to_csv("terrains.csv", index=False)
print("Terrains sauvegardés : terrains.csv")

## Résumé

| Étape | Description |
|-------|-------------|
| Dataset | 50 clients × 8 terrains, scores 1-5 basés sur statut réservation |
| Modèle | SVD (Collaborative Filtering) via scikit-surprise |
| Évaluation | RMSE et MAE via cross-validation 5-fold |
| Filtre | Seuls les terrains disponibles sont recommandés |
| Export | Modèle sauvegardé en .pkl pour le web service |

Le web service Flask (`app.py`) charge ce modèle et expose l'endpoint `/recommend`.